# TC1 · SECOP Data Pipeline

## Misión profesional

**Curso:** Big Data · Maestría en Analítica de Datos  
**Modalidad:** parejas · **100 puntos** · **trabajo dentro y fuera de clase**

Un equipo de analítica necesita construir un pipeline reproducible sobre SECOP II. El producto debe adquirir datos desde una API real, conservar trazabilidad, resistir fallos transitorios, demostrar equivalencia entre una estrategia secuencial y otra concurrente y reutilizar el mismo snapshot en modelos documental, wide-column y de grafo.

> **La evaluación se centra en resultados verificables y decisiones de ingeniería.** No se califica memorizar sintaxis ni seguir una receta.

El LMS contiene el briefing y la rúbrica. Este notebook es el **único entorno de trabajo**.

```text
SECOP II API
   ↓
adquisición reproducible
   ↓
secuencial ↔ concurrente
   ↓
RAW + calidad + trazabilidad
   ↓
Atlas idempotente
   ↓
producto analítico
   ↓
Cassandra + Neo4j
   ↓
decisiones + evidencia verificable
```

### Dedicación esperada

Este no es un ejercicio de 180 minutos. Se inicia en clase y se completa en casa.

**Dedicación mínima prevista por grupo: 6 horas de trabajo efectivo**, distribuidas entre adquisición, depuración, modelado, ejecución en servicios reales, validación, corrección y preparación de evidencia.

El tiempo no otorga puntos por sí mismo: la exigencia de 6+ horas describe la profundidad esperada del producto.

## Rúbrica por capacidades

| Etapa | Capacidad que debe demostrar | Puntos |
|---|---|---:|
| E1 | adquirir SECOP de forma reproducible, resiliente y concurrente | 25 |
| E2 | diseñar y operar un modelo documental idempotente en Atlas | 25 |
| E3 | convertir el snapshot en un producto analítico verificable | 10 |
| E4 | diseñar Cassandra desde el patrón de consulta | 15 |
| E5 | modelar y consultar contexto relacional con Neo4j | 15 |
| E6 | justificar decisiones, límites y reproducibilidad | 10 |
| **Total** | | **100** |

### Reglas de evaluación

- El *speedup* no da puntos por sí mismo.
- La ruta secuencial y la concurrente deben representar el mismo snapshot.
- Una segunda carga en Atlas no debe duplicar documentos.
- El validador entrega feedback por criterio fallido.
- Exponer secretos es un **bloqueo de entrega**: corrija el artefacto y vuelva a validar.

## Plan de trabajo recomendado · 6–8 horas por grupo

No tienen que terminarlo en una sola sesión. Organicen el trabajo y conserven el mismo `PAREJA_ID` durante todo el proyecto.

| Tramo | Trabajo | Dedicación orientativa |
|---|---|---:|
| 1 | contrato de datos, SoQL, pruebas y adquisición secuencial | 60–75 min |
| 2 | concurrencia, retries, benchmark, hashes y calidad | 75–90 min |
| 3 | integración + modelo documental + Atlas idempotente | 90–120 min |
| 4 | producto analítico + Cassandra query-first | 60–75 min |
| 5 | Neo4j + contraste relacional | 60–75 min |
| 6 | decisiones, microdefensa, validación y paquete final | 45–60 min |

**Trabajo en casa:** completar lo que no alcance durante la sesión, volver a ejecutar desde cero, corregir fallos y preparar el paquete final.

**Evidencia obligatoria del grupo:** notebook ejecutado + `TC1_<pareja>.zip` + `manifest_tc1.json`. Un integrante carga el manifest; el LMS replica la misma calificación a todo el equipo.

In [ ]:
# Instalación mínima en Colab
!pip -q install pymongo pyarrow

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from urllib.parse import quote_plus
from getpass import getpass
import hashlib, json, math, os, random, re, time

import pandas as pd
import requests

OUT = Path("entrega_tc1")
RAW = OUT / "raw"
OUT.mkdir(exist_ok=True)
RAW.mkdir(exist_ok=True)
print("Carpeta de trabajo:", OUT.resolve())

## 0 · Identidad y partición determinística

Cada pareja recibe una ventana temporal a partir de `PAREJA_ID`. Así todos resuelven la misma competencia, pero no producen exactamente los mismos resultados.

**Completen estos datos antes de seguir.**

In [ ]:
PAREJA_ID = ""        # TODO: ejemplo "P03"
INTEGRANTE_1 = ""      # TODO
CODIGO_1 = ""          # TODO
INTEGRANTE_2 = ""      # TODO
CODIGO_2 = ""          # TODO

VENTANAS_2025 = [
    ("2025-01-01T00:00:00.000", "2025-03-01T00:00:00.000"),
    ("2025-03-01T00:00:00.000", "2025-05-01T00:00:00.000"),
    ("2025-05-01T00:00:00.000", "2025-07-01T00:00:00.000"),
    ("2025-07-01T00:00:00.000", "2025-09-01T00:00:00.000"),
    ("2025-09-01T00:00:00.000", "2025-11-01T00:00:00.000"),
    ("2025-11-01T00:00:00.000", "2026-01-01T00:00:00.000"),
]

if len(PAREJA_ID.strip()) < 3:
    print("⚠️ Completen PAREJA_ID antes de descargar.")
else:
    idx = int(hashlib.sha256(PAREJA_ID.strip().encode()).hexdigest()[:8], 16) % len(VENTANAS_2025)
    FECHA_INI, FECHA_FIN = VENTANAS_2025[idx]
    print("Ventana asignada:", FECHA_INI, "→", FECHA_FIN)

## E1 · Adquisición SECOP verificable — 25 puntos

Usarán dos fuentes oficiales:

- **Procesos SECOP II:** `p6dx-8zbt`
- **Contratos SECOP II:** `jbjy-vk9h`

La API debe recibir **solo las columnas necesarias**. Esto es *query pushdown*: filtrar y proyectar antes de transferir.

Un App Token es opcional. Socrata permite consultas públicas sin token, pero un token separa el throttling de la aplicación del pool compartido por IP. Nunca escriban un secreto en el notebook.

In [ ]:
# Infraestructura provista · configuración mínima
BASE = "https://www.datos.gov.co/resource"
ENDPOINTS = {
    "procesos": "p6dx-8zbt",
    "contratos": "jbjy-vk9h",
}

SELECT_PROCESOS = [
    "id_del_proceso","entidad","nit_entidad","departamento_entidad","ciudad_entidad",
    "fecha_de_publicacion","precio_base","modalidad_de_contratacion",
    "respuestas_al_procedimiento","estado_del_procedimiento","adjudicado",
    "nombre_del_proveedor_adjudicado","nit_del_proveedor_adjudicado","urlproceso"
]

SELECT_CONTRATOS = [
    "proceso_de_compra","id_contrato","estado_contrato","tipo_de_contrato",
    "modalidad_de_contratacion","fecha_de_firma","proveedor_adjudicado",
    "documento_proveedor","valor_del_contrato"
]

APP_TOKEN = ""  # opcional; nunca lo incluya en los entregables
HEADERS = {"Accept": "application/json"}
if APP_TOKEN:
    HEADERS["X-App-Token"] = APP_TOKEN

data_contract = {
    "procesos": {"id": ENDPOINTS["procesos"], "grain": "proceso de contratación", "fields": SELECT_PROCESOS},
    "contratos": {"id": ENDPOINTS["contratos"], "grain": "contrato electrónico", "fields": SELECT_CONTRATOS},
    "join": {"procesos.id_del_proceso": "contratos.proceso_de_compra"},
    "window": {"start": globals().get("FECHA_INI"), "end": globals().get("FECHA_FIN")},
}
(OUT / "00_dataset_contract.json").write_text(json.dumps(data_contract, ensure_ascii=False, indent=2), encoding="utf-8")
data_contract

In [ ]:
# Infraestructura provista · cliente HTTP resiliente
RETRY_STATUS = {429, 500, 502, 503, 504}

def request_json(url, params, *, timeout=45, max_attempts=5, base_backoff=1.0):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        started = time.perf_counter()
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=timeout)
            elapsed = time.perf_counter() - started
            if r.status_code in RETRY_STATUS:
                retry_after = r.headers.get("Retry-After")
                wait = float(retry_after) if retry_after and retry_after.isdigit() else base_backoff * (2 ** (attempt - 1))
                time.sleep(min(wait, 12))
                continue
            r.raise_for_status()
            return r.json(), {
                "status": r.status_code,
                "elapsed_s": round(elapsed, 4),
                "bytes": len(r.content),
                "attempts": attempt,
            }
        except requests.RequestException as exc:
            last_error = exc
            if attempt == max_attempts:
                raise
            time.sleep(min(base_backoff * (2 ** (attempt - 1)), 12))
    raise RuntimeError(last_error)

def fetch_page(endpoint_id, *, select, where, order, limit, offset):
    params = {
        "$select": ",".join(select),
        "$where": where,
        "$order": order,
        "$limit": int(limit),
        "$offset": int(offset),
    }
    rows, meta = request_json(f"{BASE}/{endpoint_id}.json", params)
    meta.update({"endpoint": endpoint_id, "offset": int(offset), "limit": int(limit), "rows": len(rows)})
    return rows, meta

### E1.1 · Primera consulta: pequeña y explicable

Antes de paralelizar, demuestren que la consulta funciona con **50 filas**. El orden debe ser estable.

**Procesos:** filtren por `fecha_de_publicacion` en su ventana.  
**Contratos:** filtren por `fecha_de_firma` en la misma ventana.

Construyan `query_plan` y prueben ambos endpoints.

In [ ]:
# Reto · plan de consulta
WHERE_PROCESOS = None
WHERE_CONTRATOS = None

query_plan = {
    "procesos": {
        "where": WHERE_PROCESOS,
        "order": "fecha_de_publicacion ASC,id_del_proceso ASC",
        "select": SELECT_PROCESOS,
    },
    "contratos": {
        "where": WHERE_CONTRATOS,
        "order": "fecha_de_firma ASC,id_contrato ASC",
        "select": SELECT_CONTRATOS,
    },
}

# Evidencia de prueba controlada: recupere hasta 50 filas por fuente.
muestra_procesos = None
muestra_contratos = None

### E1.2 · Descarga secuencial

Implementen una función que descargue páginas en orden hasta `max_rows` o hasta que la API devuelva una página incompleta.

Requisitos:

- `PAGE_SIZE = 1000`;
- registrar metadata por petición;
- no concatenar manualmente archivos;
- preservar el orden estable de la consulta.

In [ ]:
PAGE_SIZE = 1000
N_PROCESOS = 6000
N_CONTRATOS = 4000

def descargar_secuencial(endpoint_id, *, select, where, order, max_rows, page_size=PAGE_SIZE):
    """Retorne (DataFrame, metadata_requests) preservando el orden de la consulta."""
    return None, []

procesos_seq = None
meta_procesos_seq = []
tiempo_seq = None

### E1.3 · Descarga concurrente con `ThreadPoolExecutor`

Esto es I/O-bound: durante gran parte del tiempo Python está esperando red.

Implementen la misma descarga con `max_workers` controlado. En este taller solo se permite **2–6 workers**.

**No** se premia usar más workers. Se premia obtener el mismo snapshot y justificar la elección.

### Micro-lab · dos llamadas concurrentes (no evaluado)

Antes de construir la función general, observe un patrón mínimo con **dos páginas de 25 filas**. El objetivo es entender tres ideas:

1. cada llamada sigue siendo una petición HTTP normal;
2. el executor coordina esperas de I/O, no “acelera Python”;
3. los resultados pueden terminar en distinto orden, por eso debe conservar el `offset`.

Ejecute este ejemplo después de definir `WHERE_PROCESOS`. Luego construya su propia solución general sin copiar una lista fija de offsets.

In [ ]:
# Micro-lab provisto · patrón mínimo, no suma puntos
def demo_dos_paginas():
    if not isinstance(WHERE_PROCESOS, str) or not WHERE_PROCESOS.strip():
        raise ValueError("Defina WHERE_PROCESOS antes del micro-lab.")

    offsets = [0, 25]
    resultados = []
    with ThreadPoolExecutor(max_workers=2) as pool:
        futuros = {
            pool.submit(
                fetch_page,
                ENDPOINTS["procesos"],
                select=SELECT_PROCESOS,
                where=WHERE_PROCESOS,
                order=query_plan["procesos"]["order"],
                limit=25,
                offset=offset,
            ): offset
            for offset in offsets
        }
        for futuro in as_completed(futuros):
            offset = futuros[futuro]
            rows, meta = futuro.result()
            resultados.append((offset, rows, meta))

    resultados.sort(key=lambda x: x[0])
    return resultados

demo_resultados = demo_dos_paginas() if isinstance(WHERE_PROCESOS, str) and WHERE_PROCESOS.strip() else []
[(offset, len(rows)) for offset, rows, _ in demo_resultados]

In [ ]:
MAX_WORKERS = 4

def descargar_concurrente(endpoint_id, *, select, where, order, max_rows, page_size=PAGE_SIZE, max_workers=MAX_WORKERS):
    """Retorne (DataFrame, metadata_requests) con páginas concurrentes y resultado determinista."""
    return None, []

procesos_df = None
meta_procesos = []
tiempo_threads = None

In [ ]:
# Infraestructura provista · comparación canónica
def canonical_hash(df, key):
    if not isinstance(df, pd.DataFrame):
        return None
    ordered = df.copy().sort_values(key, kind="mergesort").reset_index(drop=True)
    payload = ordered.to_json(orient="records", force_ascii=False, date_format="iso")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

hash_seq = None
hash_threads = None

benchmark_threads = {
    "workers": MAX_WORKERS,
    "rows_sequential": 0 if procesos_seq is None else len(procesos_seq),
    "rows_threaded": 0 if procesos_df is None else len(procesos_df),
    "seconds_sequential": tiempo_seq,
    "seconds_threaded": tiempo_threads,
    "hash_sequential": hash_seq,
    "hash_threaded": hash_threads,
    "same_rows": False,
    "same_hash": False,
}
benchmark_threads

### E1.4 · Más información de SECOP: contratos

Repitan la descarga concurrente sobre `jbjy-vk9h`.

No intenten descargar millones de filas. La ventana asignada y `N_CONTRATOS` mantienen el ejercicio controlado.

Después midan la cobertura del cruce:

```
procesos.id_del_proceso == contratos.proceso_de_compra
```

In [ ]:
# Reto · segunda fuente SECOP y cobertura del cruce
contratos_df = None
meta_contratos = []
join_coverage = None

### E1.5 · Calidad, trazabilidad y RAW

Guarden los snapshots crudos como Parquet y produzcan:

- `01_acquisition_manifest.json`;
- `01_benchmark_threads.json`;
- `01_quality_report.json`.

La trazabilidad debe incluir endpoint, filtros, orden, filas, bytes, reintentos, workers y timestamp UTC. **Nunca** incluyan App Token, URI de Atlas, usuario o contraseña.

In [ ]:
def profile_quality(df, key):
    return {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "duplicate_keys": int(df[key].duplicated().sum()) if key in df else None,
        "null_key": int(df[key].isna().sum()) if key in df else None,
        "null_pct": {c: round(float(df[c].isna().mean() * 100), 2) for c in df.columns[:20]},
    }

# TODO: guardar procesos_df y contratos_df en RAW como Parquet
# procesos_df.to_parquet(...)
# contratos_df.to_parquet(...)

quality_report = {
    "procesos": {} if procesos_df is None else profile_quality(procesos_df, "id_del_proceso"),
    "contratos": {} if contratos_df is None else profile_quality(contratos_df, "id_contrato"),
    "join_coverage": join_coverage,
}

acquisition_manifest = {
    "schema": "2026-09-26-secoppipeline",
    "queried_at_utc": datetime.now(timezone.utc).isoformat(),
    "pair": PAREJA_ID,
    "window": {"start": globals().get("FECHA_INI"), "end": globals().get("FECHA_FIN")},
    "workers": MAX_WORKERS,
    "datasets": {
        "procesos": {"id": ENDPOINTS["procesos"], "rows": 0 if procesos_df is None else len(procesos_df), "requests": len(meta_procesos)},
        "contratos": {"id": ENDPOINTS["contratos"], "rows": 0 if contratos_df is None else len(contratos_df), "requests": len(meta_contratos)},
    },
    "benchmark": benchmark_threads,
}

(OUT / "01_acquisition_manifest.json").write_text(json.dumps(acquisition_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "01_benchmark_threads.json").write_text(json.dumps(benchmark_threads, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "01_quality_report.json").write_text(json.dumps(quality_report, ensure_ascii=False, indent=2), encoding="utf-8")
quality_report

## E2 · Modelo documental + MongoDB Atlas — 25 puntos

Ahora conviertan el snapshot en documentos útiles.

El modelo debe contener, como mínimo:

```json
{
  "id_proceso": "...",
  "entidad": {...},
  "proceso": {...},
  "proveedor_adjudicado": {...},
  "contratos_resumen": {
    "cantidad": 2,
    "valor_total": 123000000,
    "estados": ["En ejecución"]
  },
  "metadata_ingesta": {
    "dataset": "p6dx-8zbt",
    "ventana": "...",
    "pareja_id": "..."
  }
}
```

No copien todas las columnas sin criterio.

In [ ]:
# Reto · integración y modelo documental
# Contrato de salida:
# - historico: una fila por id_proceso
# - documentos: lista de documentos anidados coherentes con el briefing

historico = None
documentos = None

### E2.2 · Atlas real e idempotencia

Conéctense con `getpass()`. No guarden credenciales.

En vez de `delete_many()+insert_many()`, usen:

- índice único por `id_proceso`;
- `bulk_write`;
- `UpdateOne(..., upsert=True)`.

Ejecuten la carga **dos veces** y demuestren que el número de documentos no aumenta.

In [ ]:
from pymongo import MongoClient, UpdateOne, ASCENDING, DESCENDING

atlas_ping = False
atlas_server_version = None
coleccion = None
atlas_idempotencia = {}
atlas_indexes = []

# Reto:
# construya una carga idempotente sobre la base "tc1_bigdata".
# La segunda ejecución debe conservar exactamente el mismo número de documentos.
# Cree un índice único para la identidad del proceso y al menos un índice adicional
# justificado por una consulta del taller.

### E2.3 · Consultas documentales

Construyan:

**Consulta A:** procesos con `precio_base > 0` y al menos un contrato asociado.  
**Consulta B:** top 10 por valor total de contratos, desempate por `id_proceso ASC`.

La evidencia debe guardar filtros/proyección/resultado, no credenciales.

In [ ]:
filtro_a = None       # TODO
resultado_a = None    # TODO: count_documents

filtro_b = None       # TODO
proyeccion_b = None   # TODO
resultado_b = None    # TODO: lista top 10

atlas_resultados = {
    "carga": {
        "documentos": None if coleccion is None else coleccion.count_documents({}),
        "server_version": atlas_server_version,
        "idempotencia": atlas_idempotencia,
        "indices": atlas_indexes,
    },
    "consulta_a": {"filtro": filtro_a, "resultado": resultado_a},
    "consulta_b": {"filtro": filtro_b, "proyeccion": proyeccion_b, "resultado": resultado_b},
}
(OUT / "02_atlas_evidence.json").write_text(json.dumps(atlas_resultados, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

## E3 · Producto analítico — 10 puntos

Construyan desde **Atlas**, no desde una copia manual, una bandeja de revisión priorizada.

Criterios mínimos:

- proceso con precio positivo;
- al menos un contrato asociado;
- ordenar por `valor_contratos DESC`, luego `id_proceso ASC`;
- máximo 100 filas.

La bandeja **no prueba fraude**: solo prioriza revisión.

In [ ]:
pipeline_bandeja = None       # TODO: pipeline MongoDB
bandeja_documentos = None      # TODO
bandeja_historica = None       # TODO: DataFrame

# TODO: guardar OUT / "03_bandeja_historica.csv"

## E4 · Cassandra query-first — 15 puntos

Pregunta operacional:

> Dado un `anio` y un `departamento`, mostrar hasta 10 procesos priorizados comenzando por mayor `valor_contratos`; en empate usar `id_proceso ASC`.

Diseñen:

- `bandeja_cassandra`;
- `cql_create`;
- `consulta_cassandra_simulada(df, anio, departamento, n=10)`;
- `particion_prueba`;
- `top10_cassandra`.

No usen `ALLOW FILTERING`.

In [ ]:
bandeja_cassandra = None
cql_create = ""   # TODO

def consulta_cassandra_simulada(df, anio, departamento, n=10):
    # TODO
    return None

particion_prueba = None
top10_cassandra = None

# TODO: guardar OUT / "04_modelo_cassandra.cql"

## E5 · Neo4j y contexto relacional — 15 puntos

Pregunta:

> ¿Qué proveedores conectan a la entidad con mayor número de procesos adjudicados con otras entidades dentro del snapshot de la pareja?

Construyan:

- `hist_adjudicado`;
- `nit_ancla`, `entidad_ancla`;
- `resultado_relacional`;
- `cypher_carga`, `cypher_contexto`, `cypher_compartidos`, `cypher_ranking`;
- `G = nx.DiGraph()`, `nodos_grafo`, `aristas_grafo`.

La conexión contractual **no demuestra colusión, favorecimiento o fraude**.

In [ ]:
import networkx as nx

hist_adjudicado = None
nit_ancla = None
entidad_ancla = None
resultado_relacional = None

cypher_carga = ""
cypher_contexto = ""
cypher_compartidos = ""
cypher_ranking = ""

G = nx.DiGraph()
nodos_grafo = 0
aristas_grafo = 0

# TODO: guardar
# OUT / "05_resultado_relacional.csv"
# OUT / "05_neo4j_consultas.cypher"

## E6 · Decisiones, microdefensa e informe — 10 puntos

Registren **al menos 3 decisiones propias**. Cada decisión debe contener:

- decisión;
- evidencia observada;
- alternativa;
- riesgo o límite.

Ejemplos válidos: workers, columnas seleccionadas, page size, modelo documental, índice Atlas.

El informe debe explicar lo construido y por qué.

In [ ]:
decision_log = [
    # Registre mínimo 3 decisiones reales: evidencia, alternativa y riesgo.
    # {"decision": "...", "evidence": "...", "alternative": "...", "risk": "..."},
]

(OUT / "06_decision_log.json").write_text(json.dumps(decision_log, ensure_ascii=False, indent=2), encoding="utf-8")

informe_tecnico = """
## 1. Adquisición y contrato de datos

## 2. Concurrencia, robustez y calidad

## 3. Modelo documental e idempotencia Atlas

## 4. Producto analítico y Cassandra

## 5. Neo4j, decisiones y límites

"""
# TODO: completen con resultados calculados y la afirmación explícita
# de que priorización/conexiones NO demuestran fraude.
(OUT / "06_informe_tecnico.md").write_text(informe_tecnico, encoding="utf-8")

### E6.2 · Microdefensa grupal basada en su propia ejecución

Respondan como equipo, con evidencia de sus resultados. No hay una “respuesta de plantilla”.

**Pregunta 1 — concurrencia:** con el número de workers que realmente utilizaron, ¿qué cambiarían si la API empieza a responder HTTP 429 y cómo comprobarían que el cambio no alteró el snapshot?

**Pregunta 2 — calidad:** a partir de su `join_coverage`, expliquen por qué los procesos sin contrato relacionado no deben rellenarse o inventarse y qué significa esa cobertura para el alcance del análisis.

Cada respuesta debe citar al menos un resultado propio (workers, hash, tiempos, cobertura, conteos o errores observados).

In [ ]:
defensa_grupal = {
    "pregunta_concurrencia": f"Con {MAX_WORKERS} workers, ¿qué cambiaríamos ante HTTP 429 sin alterar el snapshot?",
    "respuesta_concurrencia": "",  # respuesta del grupo
    "pregunta_calidad": f"Con join_coverage={join_coverage}, ¿qué significa la cobertura y por qué no rellenamos contratos faltantes?",
    "respuesta_calidad": "",       # respuesta del grupo
}

(OUT / "06_microdefensa_grupal.json").write_text(
    json.dumps(defensa_grupal, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
defensa_grupal

## Validación final

El validador **no compara números contra una respuesta fija** porque SECOP se actualiza. Evalúa invariantes de ingeniería y coherencia entre sus propios artefactos.

Ejecuten esta celda solo cuando hayan completado E1–E6.

In [ ]:
# Descargar el validador oficial vigente
VALIDATOR_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/utils/tc1_validator.py"
validator_text = requests.get(VALIDATOR_URL, timeout=30).text
if "def evaluar" not in validator_text:
    raise RuntimeError("No se pudo descargar el validador oficial.")
Path("tc1_validator.py").write_text(validator_text, encoding="utf-8")

import importlib.util
spec = importlib.util.spec_from_file_location("tc1_validator", "tc1_validator.py")
tc1_validator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tc1_validator)

manifest = tc1_validator.evaluar(globals())
manifest

## Entrega

**Entrega por grupo:**

1. el `.ipynb` ejecutado con salidas visibles;
2. `TC1_<pareja>.zip`, que incluye la microdefensa grupal;
3. un integrante carga `manifest_tc1.json` en el LMS.

El LMS registra **una sola calificación del grupo** y la replica a todos sus integrantes. No carguen el mismo manifest desde las dos cuentas.

**No entreguen:** App Token, URI de Atlas, usuario, contraseña, capturas como sustituto del código ni resultados escritos manualmente.